**Table of contents**<a id='toc0_'></a>    
- [LSTM Implementation Checks](#toc1_)    
  - [Import PyTorch and Define Dimensions](#toc1_1_)    
  - [Define LSTM Dimensions and Initial States](#toc1_2_)    
  - [Compute One LSTM Step Manually](#toc1_3_)    
  - [Compare the Manual Computation with `nn.LSTMCell`](#toc1_4_)    
  - [Compare Sequence Processing with `nn.LSTM`](#toc1_5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[LSTM Implementation Checks](#toc0_)

This notebook compares a manual LSTM computation with PyTorch's `nn.LSTMCell` and `nn.LSTM`.

## <a id='toc1_1_'></a>[Import PyTorch and Define Dimensions](#toc0_)

In [1]:
import torch
import torch.nn as nn

## <a id='toc1_2_'></a>[Define LSTM Dimensions and Initial States](#toc0_)

In [2]:
embed_dim = 3
hidden_dim = 2

## <a id='toc1_3_'></a>[Compute One LSTM Step Manually](#toc0_)

In [3]:
x_t = torch.tensor([[0.5, -0.2, 0.1]])
h_prev = torch.tensor([[0.1, 0.2]])
c_prev = torch.tensor([[0.0, 0.0]])

W_xi, W_hi = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xf, W_hf = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xc, W_hc = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))
W_xo, W_ho = torch.rand((embed_dim, hidden_dim)), torch.rand((hidden_dim, hidden_dim))

b_i = torch.rand(hidden_dim) * 0.1
b_f = torch.rand(hidden_dim) * 0.1
b_c = torch.rand(hidden_dim) * 0.1
b_o = torch.rand(hidden_dim) * 0.1

W_x = torch.cat([W_xi.T, W_xf.T, W_xc.T, W_xo.T], dim=0)
W_h = torch.cat([W_hi.T, W_hf.T, W_hc.T, W_ho.T], dim=0)
B = torch.cat([b_i.T, b_f.T, b_c.T, b_o.T], dim=0)

sum = x_t @ W_x.T + h_prev @ W_h.T + B  # [i, f, c, o]

i_t, f_t, c_tilde_t, o_t = torch.chunk(sum, 4, dim=1)

i_t = torch.sigmoid(i_t)
f_t = torch.sigmoid(f_t)
c_tilde_t = torch.tanh(c_tilde_t)
o_t = torch.sigmoid(o_t)


c_t = f_t * c_prev + i_t * c_tilde_t
h_t = o_t * torch.tanh(c_t)

print(f"W_x.shape = {W_x.shape}")
print(f"W_h.shape = {W_h.shape}")
print(f"B.shape = {B.shape}")

print(c_t)
print(h_t)

W_x.shape = torch.Size([8, 3])
W_h.shape = torch.Size([8, 2])
B.shape = torch.Size([8])
tensor([[0.1862, 0.3437]])
tensor([[0.0940, 0.2189]])


C:\Users\Huy\AppData\Local\Temp\ipykernel_10976\488287044.py:17: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4323.)
  B = torch.cat([b_i.T, b_f.T, b_c.T, b_o.T], dim=0)


## <a id='toc1_4_'></a>[Compare the Manual Computation with `nn.LSTMCell`](#toc0_)

## <a id='toc1_5_'></a>[Compare Sequence Processing with `nn.LSTM`](#toc0_)

In [4]:
embed_dim = 3
hidden_dim = 2

lstm_cell = nn.LSTMCell(embed_dim, hidden_dim)

with torch.no_grad(): # [i, f, c, o]
    lstm_cell.weight_ih.copy_(torch.cat([W_xi.T, W_xf.T, W_xc.T, W_xo.T], dim=0))
    lstm_cell.weight_hh.copy_(torch.cat([W_hi.T, W_hf.T, W_hc.T, W_ho.T], dim=0))
    lstm_cell.bias_ih.copy_(torch.cat([b_i, b_f, b_c, b_o], dim=0))
    lstm_cell.bias_hh.zero_()

print(f"lstm_cell.weight_ih.shape = {lstm_cell.weight_ih.shape}")
print(f"lstm_cell.weight_hh.shape = {lstm_cell.weight_hh.shape}")
print(f"lstm_cell.bias_ih.shape = {lstm_cell.bias_ih.shape}")
print(f"lstm_cell.bias_hh.shape = {lstm_cell.bias_hh.shape}")

h_t_pytorch, c_t_pytorch = lstm_cell(x_t, (h_prev, c_prev))
print(f"h_t_pytorch = {h_t_pytorch}")
print(f"c_t_pytorch = {c_t_pytorch}")

lstm_cell.weight_ih.shape = torch.Size([8, 3])
lstm_cell.weight_hh.shape = torch.Size([8, 2])
lstm_cell.bias_ih.shape = torch.Size([8])
lstm_cell.bias_hh.shape = torch.Size([8])
h_t_pytorch = tensor([[0.0940, 0.2189]], grad_fn=<MulBackward0>)
c_t_pytorch = tensor([[0.1862, 0.3437]], grad_fn=<AddBackward0>)


In [5]:
seq_len = 4
embed_dim = 3
hidden_dim = 2

x_seq = torch.rand(1, seq_len, embed_dim) # (batch, sequence len, embed_dim)

lstm_multi = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
with torch.no_grad():
    lstm_multi.weight_ih_l0.copy_(lstm_cell.weight_ih)
    lstm_multi.weight_hh_l0.copy_(lstm_cell.weight_hh)
    lstm_multi.bias_ih_l0.copy_(lstm_cell.bias_ih)
    lstm_multi.bias_hh_l0.copy_(lstm_cell.bias_hh)

outputs, (h_n, c_n) = lstm_multi(x_seq)
print(h_n)
h, c = torch.zeros(1, hidden_dim), torch.zeros(1, hidden_dim)
print(outputs)


tensor([[[0.7538, 0.6207]]], grad_fn=<StackBackward0>)
tensor([[[0.0646, 0.0168],
         [0.2361, 0.1536],
         [0.4129, 0.4422],
         [0.7538, 0.6207]]], grad_fn=<TransposeBackward0>)


In [6]:

manual_outputs = []
for t in range(seq_len):
    h, c = lstm_cell(x_seq[:, t, :], (h, c))
    manual_outputs.append(h)
manual_outputs = torch.stack(manual_outputs, dim=1)

print(manual_outputs)


tensor([[[0.0646, 0.0168],
         [0.2361, 0.1536],
         [0.4129, 0.4422],
         [0.7538, 0.6207]]], grad_fn=<StackBackward0>)


In [7]:
hidden_dim = 100
onehot_dim = 10000

a = torch.rand(hidden_dim, 1)
x = torch.rand(onehot_dim, 1)
Waa = torch.rand(hidden_dim, hidden_dim)
Wax = torch.rand(hidden_dim, onehot_dim)
bias = torch.rand(hidden_dim, 1)

a_1 = torch.sigmoid(Waa @ a + Wax @ x + bias)

Wa = torch.cat([Waa, Wax], dim=1)
X = torch.cat([a, x], dim=0)

a_1_test = torch.sigmoid(Wa @ X + bias)

assert a_1.shape == a_1_test.shape, "Differ shape"
print("Same shape")

Same shape


In [16]:
Waa = torch.rand(2, 2)
Wax = torch.rand(2, 10)

print(Waa)
print(Wax)
Wa = torch.cat([Waa, Wax], dim=1)
print(Wa)

tensor([[0.4423, 0.5479],
        [0.1546, 0.6880]])
tensor([[0.0393, 0.2604, 0.5939, 0.4594, 0.4568, 0.9195, 0.9367, 0.0536, 0.7335,
         0.8296],
        [0.4645, 0.9348, 0.6867, 0.8605, 0.4415, 0.7190, 0.8482, 0.5293, 0.4904,
         0.7548]])
tensor([[0.4423, 0.5479, 0.0393, 0.2604, 0.5939, 0.4594, 0.4568, 0.9195, 0.9367,
         0.0536, 0.7335, 0.8296],
        [0.1546, 0.6880, 0.4645, 0.9348, 0.6867, 0.8605, 0.4415, 0.7190, 0.8482,
         0.5293, 0.4904, 0.7548]])
